# 1. Project Overview

MermaidGenerate is a Google Colab-compatible local AI web app for generating Mermaid **Mind Map** and **Venn Diagram** code. The primary runtime is Hugging Face **Transformers + PyTorch**, not paid APIs. Fine-tuning is supported through **LoRA**, **QLoRA 4-bit**, and **Full Fine-Tuning** paths using PEFT.

Default base model: `TinyLlama/TinyLlama-1.1B-Chat-v1.0`.

The Gradio app provides two tabs: **Generator Mermaid** and **Dataset & Fine-Tuning**.

## How to run in Colab

1. Open this notebook in Google Colab.
2. Select **Runtime > Change runtime type > GPU** when you want model loading or LoRA/QLoRA training.
3. Run cells from top to bottom.
4. Validate a dataset before training.
5. Launch the Gradio app and use the two tabs for demo.
6. Download adapter ZIP after a successful LoRA/QLoRA run.

## Demo flow for lecturer

1. Show dependency installation and GPU check.
2. Load helper modules and default model configuration.
3. Run sample Mind Map and Venn validation.
4. Launch Gradio.
5. Upload curated dataset.
6. Start a small LoRA smoke training only if GPU is available.
7. Refresh training status until adapter output appears.
8. Generate a before/after Mermaid sample and show validation result.


# 2. Install Dependencies

This cell is Colab-friendly. If the notebook is opened without the repository files, it clones the GitHub repo first, then installs `requirements.txt`. In local usage, it uses the current project folder.

## Restart runtime note

If pip upgrades packages that were already imported in the current Python runtime, restart the Colab runtime and run the notebook again from the first cell. This prevents mixed old/new module versions.


In [ ]:
from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/Justindwinata/MermaidGenerate.git"
PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "src" / "mermaid_generate").exists():
    if PROJECT_DIR.name != "MermaidGenerate":
        !git clone {REPO_URL} MermaidGenerate
        os.chdir("MermaidGenerate")
        PROJECT_DIR = Path.cwd()

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

%pip install -q -r requirements.txt
print("Project directory:", PROJECT_DIR)
print("Source directory:", SRC_DIR)


# 3. Import Libraries

Core imports for dataset validation, Mermaid preview, Transformers/PyTorch inference, fine-tuning, adapter activation, evaluation, and Gradio.


In [ ]:
from pathlib import Path
import json
import time

from mermaid_generate.config import DEFAULT_MODEL_ID, MERMAID_JS_VERSION
from mermaid_generate.dataset_validator import validate_dataset_file
from mermaid_generate.inference import GenerationSettings, generate_mermaid
from mermaid_generate.mermaid_preview import build_mermaid_preview_html
from mermaid_generate.mermaid_validator import validate_mermaid_code, postprocess_mermaid_output
from mermaid_generate.model_loader import ACTIVE_MODEL_STATE, load_base_model, load_adapter, load_full_model
from mermaid_generate.training import FineTuningConfig, run_fine_tuning
from mermaid_generate.training_manager import TRAINING_MANAGER
from mermaid_generate.adapter_manager import current_adapter_metadata, activate_training_result
from mermaid_generate.evaluation import evaluate_predictions, run_manual_test_set

print("Mermaid.js pinned version:", MERMAID_JS_VERSION)
print("Default model:", DEFAULT_MODEL_ID)


# 3.1 GPU and Runtime Check

Use this cell before model loading or training. CPU inference may work but can be slow. LoRA/QLoRA training should be run on GPU. Full fine-tuning may fail on limited GPU memory. QLoRA requires compatible CUDA and bitsandbytes.


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime:", torch.version.cuda)
    print("BF16 supported:", torch.cuda.is_bf16_supported())
else:
    print("No CUDA GPU detected. Use CPU only for lightweight validation/demo, or switch Colab runtime to GPU for training.")


# 4. Configuration

Adjust these values before loading the model or starting fine-tuning. TinyLlama is used because it is local-friendly, LLaMA-family, Transformers/PyTorch-compatible, and suitable for LoRA/QLoRA experiments.


In [ ]:
MODEL_ID = DEFAULT_MODEL_ID
USE_4BIT_FOR_BASE_INFERENCE = False
LOAD_ADAPTER_PATH = None

DEFAULT_GENERATION = GenerationSettings(
    max_new_tokens=320,
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.05,
)

print({
    "MODEL_ID": MODEL_ID,
    "USE_4BIT_FOR_BASE_INFERENCE": USE_4BIT_FOR_BASE_INFERENCE,
    "LOAD_ADAPTER_PATH": LOAD_ADAPTER_PATH,
})


# 5. Reference Notebook Notes

The project inspected both reference notebooks and documented the decision in `docs/REFERENCE_NOTEBOOK_INSPECTION.md`. The first notebook is useful for Mermaid rendering and browser job patterns but uses llama.cpp/GGUF and flowcharts. The second notebook is useful for Transformers/PyTorch fine-tuning UI patterns but uses Qwen identity and flowchart-oriented logic. MermaidGenerate keeps the useful engineering patterns while targeting Mind Map and Venn only.


In [ ]:
inspection_path = Path("docs/REFERENCE_NOTEBOOK_INSPECTION.md")
print(inspection_path.read_text(encoding="utf-8")[:3000])


# 6. Dataset Upload and Validation

Accepted dataset formats:

- JSONL/JSON with `messages`
- `prompt` and `completion`
- `instruction`, optional `input`, and `output`

Every valid row is normalized to the internal schema: `id`, `diagram_type`, `prompt`, `target`, `source_format`, `language`, `domain`, and `complexity`. Training is blocked when zero valid samples exist.


In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    DATASET_PATH = Path(next(iter(uploaded))) if uploaded else None
except Exception:
    DATASET_PATH = Path("datasets/examples/mindmap_examples.jsonl") if Path("datasets/examples/mindmap_examples.jsonl").exists() else None

if DATASET_PATH:
    report = validate_dataset_file(DATASET_PATH)
    print(report.as_dict())
    VALID_DATA = report.valid_data
else:
    print("No dataset uploaded yet.")
    VALID_DATA = []


# 7. Mermaid Validator and Preview Renderer

Mind Map outputs must start with `mindmap`. Assignment-facing Venn outputs may start with `venn`; for rendering, this project converts preview code to Mermaid's supported `venn-beta` syntax. Official Mermaid docs identify Venn as `venn-beta` in Mermaid v11.12.3+, and this notebook pins Mermaid.js to a newer v11 release.


In [ ]:
mindmap_example = """mindmap
  root((Online Learning))
    Platforms
      LMS
      Video Class
    Activities
      Quiz
      Discussion"""
venn_example = """venn
  set A["Students"]
  set B["Workers"]
  union A,B["Working Students"]"""

for sample in [mindmap_example, venn_example]:
    result = validate_mermaid_code(sample)
    print(result.message())
    display({"normalized_code": result.normalized_code, "renderer_code": result.renderer_code})


# 8. Model Loading with Transformers and PyTorch

Run this cell to load the base model. If GPU memory is insufficient, the exact exception is shown; use smaller settings, LoRA/QLoRA, or Colab GPU runtime. No paid API is used.


In [ ]:
try:
    if LOAD_ADAPTER_PATH:
        state = load_adapter(LOAD_ADAPTER_PATH, model_id=MODEL_ID, use_4bit=USE_4BIT_FOR_BASE_INFERENCE)
    else:
        state = load_base_model(MODEL_ID, use_4bit=USE_4BIT_FOR_BASE_INFERENCE)
    print(state.status_text())
except Exception as exc:
    print("Model loading failed:", repr(exc))
    print("Recommendation: use GPU runtime, LoRA/QLoRA, smaller sequence length, or smaller batch size.")


# 9. Mermaid Inference

The prompt template instructs the model to return Mermaid code only. Post-processing removes markdown fences and explanation text, then validates the result.


In [ ]:
def generate_demo(prompt, diagram_type="Auto Detect"):
    return generate_mermaid(
        prompt,
        diagram_type,
        DEFAULT_GENERATION,
    )

# Example after model loading:
# result = generate_demo("Create a mind map about digital marketing for UMKM", "Mind Map")
# print(result.code)
# display(build_mermaid_preview_html(result.code))


# 10. Fine-Tuning Configuration

The same configuration object is used by the notebook and Gradio UI. Full fine-tuning can fail on limited GPU memory; the backend reports real errors and does not fake success.


In [ ]:
TRAINING_CONFIG = FineTuningConfig(
    mode="LoRA",
    model_id=MODEL_ID,
    epochs=1,
    learning_rate=2e-4,
    batch_size=1,
    gradient_accumulation=4,
    max_seq_length=1024,
    validation_split=0.1,
)
TRAINING_CONFIG.validate()
print(TRAINING_CONFIG)


# 11. LoRA / QLoRA / Full Fine-Tuning

Use this section only after `VALID_DATA` contains validated samples. Training is real. For QLoRA, CUDA and bitsandbytes compatibility are required.

## Recommended smoke demo

For assignment demo, use `configs/lora_smoke_config.json` with `datasets/curated/mixed_mindmap_venn_curated.jsonl`:

- LoRA
- max samples: 32
- epochs: 1
- batch size: 1
- gradient accumulation: 4
- max sequence length: 512
- learning rate: 2e-4
- validation split: 0.1

In the Gradio UI, upload the curated mixed dataset, validate it, enter these values, then start training from the browser. Do not run or claim success if no GPU is available.


In [ ]:
import json
from pathlib import Path

SMOKE_CONFIG_PATH = Path("configs/lora_smoke_config.json")
if SMOKE_CONFIG_PATH.exists():
    smoke_config = json.loads(SMOKE_CONFIG_PATH.read_text(encoding="utf-8"))
    print("LoRA smoke config:", smoke_config)
else:
    smoke_config = None

def run_training_once(valid_data, config):
    logs = []
    def progress(payload):
        logs.append(payload)
        print(payload)
    result = run_fine_tuning(valid_data, config, report_progress=progress)
    print(result)
    return result, logs

# Example after validating curated data and confirming GPU availability:
# small_data = VALID_DATA[: smoke_config["max_samples"]]
# smoke_training_config = FineTuningConfig(
#     mode="LoRA",
#     model_id=MODEL_ID,
#     epochs=smoke_config["epochs"],
#     learning_rate=smoke_config["learning_rate"],
#     batch_size=smoke_config["batch_size"],
#     gradient_accumulation=smoke_config["gradient_accumulation"],
#     max_seq_length=smoke_config["max_sequence_length"],
#     validation_split=smoke_config["validation_split"],
# )
# training_result, training_logs = run_training_once(small_data, smoke_training_config)


# 12. Training Manager, Logs, Cancel, and Adapter Activation

The Gradio UI uses a background training manager. It exposes honest states: idle, validating_dataset, loading_model, training, cancelling, completed, and failed. Cancellation is checked through a callback at safe training boundaries.


In [ ]:
def start_managed_training(valid_data, config):
    job_id = TRAINING_MANAGER.start(valid_data, config)
    print("Started job:", job_id)
    return job_id

def show_job(job_id):
    state = TRAINING_MANAGER.get(job_id)
    print(state.as_dict() if state else "Job not found")
    return state

# Example:
# job_id = start_managed_training(VALID_DATA, TRAINING_CONFIG)
# show_job(job_id)
# TRAINING_MANAGER.cancel(job_id)


# 13. Adapter ZIP Download

LoRA and QLoRA training outputs are saved under `outputs/adapters/<timestamp>/` and zipped for download. Full fine-tuned models are saved under `outputs/full_models/<timestamp>/`. Successful adapter/model outputs can be activated without restarting the app where resources allow.


In [ ]:
# Example after a completed TrainingResult:
# metadata = activate_training_result(training_result, model_id=MODEL_ID)
# print(metadata.as_dict())
# from google.colab import files
# if training_result.zip_path:
#     files.download(training_result.zip_path)
print(current_adapter_metadata().as_dict())


# 14. Gradio Web App

This launches the browser UI with two tabs: Generator Mermaid and Dataset & Fine-Tuning. In Colab, `share=True` creates a public temporary Gradio link.

## Demo checklist inside the UI

1. Open **Generator Mermaid** and run one Mind Map prompt.
2. Run one Venn prompt and note that preview uses Mermaid `venn-beta` internally.
3. Open **Dataset & Fine-Tuning**.
4. Upload `datasets/curated/mixed_mindmap_venn_curated.jsonl`.
5. Confirm validation reports 150 valid samples and 0 invalid samples.
6. Enter the LoRA smoke settings from `configs/lora_smoke_config.json` only if GPU is available.


In [ ]:
from app import build_app

demo = build_app()
print("Gradio app object ready. Run the next cell to launch.")


# 15. Running the Application

Run the next cell to launch. In Colab, keep the cell running while using the web app.

Expected output:

- A local Gradio URL.
- A public temporary Gradio share URL when `share=True` succeeds.
- The app opens with **Generator Mermaid** and **Dataset & Fine-Tuning** tabs.
- Dataset validation is visible before any training attempt.
- Training logs/progress update through the Refresh Status button.

If launch fails because a port is busy, change `server_port` to another value such as `7861` or restart the runtime.


In [ ]:
demo.launch(server_name="0.0.0.0", server_port=7860, share=True)


# 16. Evaluation Utilities

Evaluation does not rely only on training loss. It computes syntax validity rate, diagram type accuracy, prefix accuracy, exact match after normalization, invalid output count, and markdown fence violation count.


In [ ]:
manual_refs = [
    {
        "diagram_type": "mindmap",
        "prompt": "Create a mind map about online learning.",
        "target": mindmap_example,
    },
    {
        "diagram_type": "venn",
        "prompt": "Create a Venn diagram comparing students and workers.",
        "target": venn_example,
    },
]
manual_predictions = [mindmap_example, venn_example]
metrics = evaluate_predictions(manual_refs, manual_predictions)
print(metrics.as_dict())


# 17. Notes, Limitations, and Submission Guide

## Expected outputs

- Dataset validation summary with valid/invalid counts.
- Mermaid code that starts with `mindmap` or `venn`.
- Rendered Mermaid preview in the browser UI.
- Training status, progress, loss logs, and result summary during fine-tuning.
- Adapter directory and ZIP path after successful LoRA/QLoRA training.

## Common errors

- **No CUDA GPU detected:** switch Colab runtime to GPU for training.
- **CUDA out of memory:** reduce batch size, max sequence length, or use LoRA/QLoRA instead of Full Fine-Tuning.
- **bitsandbytes import/load error:** QLoRA needs a compatible CUDA/bitsandbytes runtime; use LoRA if compatibility fails.
- **Gradio port busy:** change the port or restart runtime.
- **Venn render error:** Venn depends on Mermaid.js v11 support; the app renders assignment-facing `venn` code internally as Mermaid `venn-beta`.

## Limitations

- Inference uses Transformers and PyTorch as required.
- llama.cpp/GGUF is optional future compatibility only unless implemented later.
- Model quality depends on dataset size and training quality.
- The example datasets are small and are not enough for strong model quality.
- Full fine-tuning may fail on limited GPU memory.
- QLoRA requires a compatible CUDA/bitsandbytes environment.
- Venn preview depends on Mermaid.js Venn support; this project pins Mermaid v11 and renders Venn via `venn-beta`.
- No paid API is used.
- Training progress and success are real, not simulated.
- Colab runtime resets can clear loaded models; save adapters/models before disconnecting.

Submission checklist: run install, validate dataset examples, load model, test inference, launch Gradio, optionally fine-tune LoRA on a larger dataset, download adapter ZIP, and submit this notebook with the repository files.
